<div align="center">

# Heavy Equipment Selling Price Prediction

### Notebook v20: Feature Mastery & Pinnacle SOTA Ensemble

---

**Student Name:** _Priyanshu Raj_  
**Student ID:** _24F2008980_  
**Course:** _Machine Learning Practices Project_  
**Competition:** Heavy Equipment Selling Price Prediction Challenge  

---

</div>


## Table of Contents

1. [Problem Statement & Feature Mastery Strategy](#1)
2. [Dataset Overview](#2)
3. [Import Libraries](#3)
4. [Load Dataset](#4)
5. [Exploratory Data Analysis](#5)
6. [Data Cleaning & Normalization](#6)
7. [Extended Feature Engineering & 50-Dim Text SVD with K-Means Clustering](#7)
8. [Hierarchical Group Statistics & Distance Metrics](#8)
9. [Compound Out-of-Fold Multi-Percentile Target Encodings](#9)
10. [Level-1 Base Models Training (10-Fold CV across 7 Model Families)](#10)
11. [Level-2 Meta-Learner Stacking & Nelder-Mead Optimization](#11)
12. [Model Performance Comparison & Feature Importance Analysis](#12)
13. [Final Submission Verification & Distribution Inspection](#13)
14. [Conclusion & Key Insights](#14)


<a id="1"></a>
# 1. Problem Statement & Feature Mastery Strategy

The goal is to predict heavy equipment auction prices (`TargetValue`). Evaluation is based on **Root Mean Squared Logarithmic Error (RMSLE)**, which corresponds to **RMSE on `log1p(TargetValue)`**.

In Notebook v20, we deploy an expanded **Feature Engineering Architecture** paired with our SOTA 7-Model Ensemble:

1. **Extended Domain & Interaction Engineering**:
   - **Depreciation Velocity**: $\frac{\text{MachineAge}}{\log1p(\text{Hours}) + 1.0}$
   - **Usage Intensity**: $\frac{\text{Hours\_log}}{\text{MachineAge}^2 + 1.0}$
   - **Configuration Complexity Score**: Count of active optional feature flags (`col1` through `col30`).
   - **Group Baseline Deviations**: Distance of machine age and hours from `(Spec_BaseClass, RegionCode)` mean baselines.

2. **50-Dimensional Unigram/Bigram Text Embeddings & K-Means Clustering**:
   - Extraction of 50 TruncatedSVD components from `Spec_FullDescriptor` using `ngram_range=(1, 2)`.
   - Unsupervised K-Means clustering ($k=15$) on text SVD components to extract discrete `Spec_Cluster_ID` features.

3. **Compound Key Multi-Percentile Target Encodings**:
   - Grouping target encodings on compound keys: `(Spec_BaseClass, SaleYear)`, `(VendorPartnerID, Spec_BaseClass)`, `(ProductConfigID, RegionCode)`.
   - Statistics: Mean, Median, Q1 (25th percentile), Q3 (75th percentile), P10, P90, and Std with Bayesian smoothing.

4. **Hierarchical Multi-Level Group Statistics**:
   - Aggregations (`mean`, `std`, `min`, `max`) for continuous variables across multi-granularity categorical keys.

5. **SOTA 7-Model Family Ensemble (10-Fold CV)**:
   - LightGBM, CatBoost, XGBoost, ExtraTrees, Ridge, Lasso, ElasticNet with Dual Level-2 Meta-Learner Stacking.


<a id="2"></a>
# 2. Dataset Overview

- **Train Set:** 138,701 rows × 50 columns.
- **Test Set:** 15,000 rows × 49 columns.
- **Target Variable:** `TargetValue` (evaluated on $\log1p$ transformation).


<a id="3"></a>
# 3. Import Libraries


In [ ]:
import os
import re
import time
import warnings
from scipy.optimize import minimize

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ All Feature Mastery libraries successfully imported!")


<a id="4"></a>
# 4. Load Dataset


In [ ]:
data_dir = "."
target_dir_root = "/kaggle/input"

if os.path.exists(target_dir_root):
    for dirpath, _, filenames in os.walk(target_dir_root):
        if "train.csv" in filenames:
            data_dir = dirpath
            break
elif os.path.exists("data"):
    data_dir = "data"
elif os.path.exists("e:/HeavyEquipmentML/data"):
    data_dir = "e:/HeavyEquipmentML/data"

print(f"📂 Data Directory Path: {data_dir}")

df_train_raw = pd.read_csv(os.path.join(data_dir, "train.csv"), low_memory=False)
df_test_raw = pd.read_csv(os.path.join(data_dir, "test.csv"), low_memory=False)
sub_template = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

print(f"📊 Raw Train Shape : {df_train_raw.shape[0]:,} rows × {df_train_raw.shape[1]} columns")
print(f"📊 Raw Test Shape  : {df_test_raw.shape[0]:,} rows × {df_test_raw.shape[1]} columns")
print(f"📊 Submission Shape: {sub_template.shape[0]:,} rows × {sub_template.shape[1]} columns")


<a id="5"></a>
# 5. Exploratory Data Analysis (EDA)


In [ ]:
log_target_val = np.log1p(df_train_raw["TargetValue"].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_train_raw["TargetValue"], kde=True, ax=axes[0], color="royalblue", bins=50)
axes[0].set_title(f"Original Target Value (Skew: {df_train_raw['TargetValue'].skew():.2f})")

sns.histplot(log_target_val, kde=True, ax=axes[1], color="teal", bins=50)
axes[1].set_title(f"log1p Transformed Target (Skew: {log_target_val.skew():.2f})")
plt.tight_layout()
plt.show()


<a id="6"></a>
# 6. Data Cleaning & Normalization


In [ ]:
def clean_data(train_df, test_df):
    df_tr = train_df.copy()
    df_te = test_df.copy()

    drop_cols = ["InventoryGroupDescription", "TransactionID"]
    df_tr.drop(columns=drop_cols, inplace=True, errors="ignore")
    df_te.drop(columns=drop_cols, inplace=True, errors="ignore")

    for col in ["YearMade", "ManufactureYear"]:
        if col in df_tr.columns:
            df_tr.loc[df_tr[col] == 1001, col] = np.nan
            df_te.loc[df_te[col] == 1001, col] = np.nan

    hours_col = "MeterHours" if "MeterHours" in df_tr.columns else "OperationalHoursMeter"
    for df in (df_tr, df_te):
        df["Hours_missing"] = (df[hours_col].isnull() | (df[hours_col] == 0.0)).astype(int)
        df.loc[df[hours_col] == 0.0, hours_col] = np.nan
        df["Hours_log"] = np.log1p(df[hours_col])
        if hours_col != "Hours_log":
            df.drop(columns=[hours_col], inplace=True, errors="ignore")

    df_tr["TransactionDate"] = pd.to_datetime(df_tr["TransactionDate"])
    df_te["TransactionDate"] = pd.to_datetime(df_te["TransactionDate"])

    return df_tr, df_te

train_prepped, test_prepped = clean_data(df_train_raw, df_test_raw)
print("✅ Data cleaned successfully!")


<a id="7"></a>
# 7. Extended Feature Engineering & 50-Dim Text SVD with K-Means Clustering


In [ ]:
def engineer_features(train_df, test_df):
    df_tr = train_df.copy()
    df_te = test_df.copy()

    mfg_col = "YearMade" if "YearMade" in df_tr.columns else "ManufactureYear"
    col_flag_cols = [c for c in df_tr.columns if c.startswith("col")]

    for dataset in (df_tr, df_te):
        t_date = dataset["TransactionDate"]
        dataset["SaleYear"] = t_date.dt.year
        dataset["SaleMonth"] = t_date.dt.month
        dataset["SaleDay"] = t_date.dt.day
        dataset["SaleDayOfWeek"] = t_date.dt.dayofweek
        dataset["SaleWeek"] = t_date.dt.isocalendar().week.astype(int)
        dataset["SaleQuarter"] = t_date.dt.quarter
        dataset["SaleElapsedDays"] = (t_date - pd.Timestamp("1990-01-01")).dt.days

        year_made = pd.to_numeric(dataset[mfg_col], errors="coerce").fillna(2000)
        dataset["MachineAge"] = (dataset["SaleYear"] - year_made).clip(lower=0)
        dataset["MachineAge_Sq"] = dataset["MachineAge"] ** 2
        dataset["MachineAge_Cubed"] = dataset["MachineAge"] ** 3

        # Advanced Interaction & Domain Ratios
        dataset["AgeXHours"] = dataset["MachineAge"] * dataset["Hours_log"]
        dataset["HoursPerYear"] = dataset["Hours_log"] / (dataset["MachineAge"] + 1.0)
        dataset["Depreciation_Velocity"] = dataset["MachineAge"] / (dataset["Hours_log"] + 1.0)
        dataset["Usage_Intensity"] = dataset["Hours_log"] / (dataset["MachineAge_Sq"] + 1.0)
        dataset["Depreciation_Curve"] = dataset["MachineAge"] / (np.sqrt(dataset["Hours_log"] + 1.0))
        dataset["Hours_Sq"] = dataset["Hours_log"] ** 2

        # Configuration Complexity & Option Flags
        dataset["Col_Flags_Sum"] = dataset[col_flag_cols].notnull().sum(axis=1) if col_flag_cols else 0
        
        # Spec Text Descriptor Features
        spec_str = dataset["Spec_FullDescriptor"].fillna("").astype(str)
        dataset["Desc_Len"] = spec_str.str.len()
        dataset["Desc_Words"] = spec_str.str.split().str.len()

        # Composite Categorical Tokens
        base_class = dataset["Spec_BaseClass"].astype(str) if "Spec_BaseClass" in dataset.columns else pd.Series("", index=dataset.index)
        prod_cfg = dataset["ProductConfigID"].astype(str) if "ProductConfigID" in dataset.columns else pd.Series("", index=dataset.index)
        region = dataset["RegionCode"].astype(str) if "RegionCode" in dataset.columns else pd.Series("", index=dataset.index)
        vendor = dataset["VendorPartnerID"].astype(str) if "VendorPartnerID" in dataset.columns else pd.Series("", index=dataset.index)

        dataset["Cat_Prod_Region"] = prod_cfg + "_" + region
        dataset["Cat_Prod_Year"] = prod_cfg + "_" + dataset["SaleYear"].astype(str)
        dataset["Cat_Base_Region"] = base_class + "_" + region
        dataset["Cat_Vendor_Base"] = vendor + "_" + base_class
        dataset["Cat_Base_Year"] = base_class + "_" + dataset["SaleYear"].astype(str)

        # Model Token Parsing
        dataset["ModelToken"] = spec_str.str.extract(r"^([A-Z0-9]+)")[0].fillna("_Missing_")
        dataset["ModelSeriesPrefix"] = spec_str.str.extract(r"^([A-Za-z]+)")[0].fillna("_Missing_")

        # Functional classification regex
        func_str = dataset["FunctionalClassification"].fillna("").astype(str) if "FunctionalClassification" in dataset.columns else pd.Series("", index=dataset.index)
        dataset["Capacity_Tons_Mid"] = (func_str.str.extract(r"(\d+\.?\d*)\s*to\s*\d+\.?\d*\s*Metric Tons")[0].astype(float) + func_str.str.extract(r"\d+\.?\d*\s*to\s*(\d+\.?\d*)\s*Metric Tons")[0].astype(float)) / 2.0
        dataset["Power_HP_Mid"] = (func_str.str.extract(r"(\d+\.?\d*)\s*to\s*\d+\.?\d*\s*Horsepower")[0].astype(float) + func_str.str.extract(r"\d+\.?\d*\s*to\s*(\d+\.?\d*)\s*Horsepower")[0].astype(float)) / 2.0

    # Frequency & Rank Encodings
    freq_cols = ["Spec_BaseClass", "VendorPartnerID", "ProductConfigID", "ModelToken", "Cat_Prod_Region", "Cat_Vendor_Base"]
    for col in freq_cols:
        freq_map = pd.concat([df_tr[col], df_te[col]], axis=0).value_counts().to_dict()
        df_tr[f"{col}_Freq"] = df_tr[col].map(freq_map)
        df_te[f"{col}_Freq"] = df_te[col].map(freq_map)
        df_tr[f"{col}_LogFreq"] = np.log1p(df_tr[f"{col}_Freq"])
        df_te[f"{col}_LogFreq"] = np.log1p(df_te[f"{col}_Freq"])

    # 50-Dimension TF-IDF + TruncatedSVD Text Embeddings
    desc_tr = df_tr["Spec_FullDescriptor"].fillna("").astype(str)
    desc_te = df_te["Spec_FullDescriptor"].fillna("").astype(str)

    tfidf = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2))
    svd = TruncatedSVD(n_components=50, random_state=42)

    tfidf_tr = tfidf.fit_transform(desc_tr)
    tfidf_te = tfidf.transform(desc_te)

    svd_tr = svd.fit_transform(tfidf_tr)
    svd_te = svd.transform(tfidf_te)

    for i in range(50):
        df_tr[f"Text_SVD_{i}"] = svd_tr[:, i]
        df_te[f"Text_SVD_{i}"] = svd_te[:, i]

    # K-Means Spec Descriptor Clustering (k=15)
    kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
    df_tr["Spec_Cluster_ID"] = kmeans.fit_predict(svd_tr)
    df_te["Spec_Cluster_ID"] = kmeans.predict(svd_te)

    return df_tr, df_te

df_tr_fe, df_te_fe = engineer_features(train_prepped, test_prepped)
print(f"✅ Extended Feature Engineering complete! Shapes: Train {df_tr_fe.shape}, Test {df_te_fe.shape}")


<a id="8"></a>
# 8. Hierarchical Group Statistics & Group Baseline Deviations


In [ ]:
def add_group_statistics(df_tr, df_te):
    df_tr_out = df_tr.copy()
    df_te_out = df_te.copy()

    group_keys = [
        ["Spec_BaseClass"],
        ["ProductConfigID"],
        ["Cat_Base_Region"],
        ["Cat_Vendor_Base"]
    ]
    target_vars = ["Hours_log", "MachineAge"]

    for keys in group_keys:
        group_name = "_".join(keys)
        for var in target_vars:
            # Combined data baseline
            comb = pd.concat([df_tr_out[keys + [var]], df_te_out[keys + [var]]], axis=0)
            stats = comb.groupby(keys)[var].agg(["mean", "std"]).reset_index()
            stats.columns = keys + [f"{group_name}_{var}_mean", f"{group_name}_{var}_std"]

            df_tr_out = df_tr_out.merge(stats, on=keys, how="left")
            df_te_out = df_te_out.merge(stats, on=keys, how="left")

            # Calculate explicit deviation from group mean baseline
            df_tr_out[f"{group_name}_{var}_diff"] = df_tr_out[var] - df_tr_out[f"{group_name}_{var}_mean"]
            df_te_out[f"{group_name}_{var}_diff"] = df_te_out[var] - df_te_out[f"{group_name}_{var}_mean"]

    return df_tr_out, df_te_out

df_tr_stat, df_te_stat = add_group_statistics(df_tr_fe, df_te_fe)
print(f"✅ Group statistics added! Shapes: Train {df_tr_stat.shape}, Test {df_te_stat.shape}")


<a id="9"></a>
# 9. Compound Out-of-Fold Multi-Percentile Target Encodings


In [ ]:
def apply_oof_target_encoding(train_df, test_df, cat_cols, target_col="TargetValue", n_splits=10, smoothing=10.0):
    df_tr = train_df.copy()
    df_te = test_df.copy()

    target_log = np.log1p(df_tr[target_col].clip(lower=0))
    global_mean = target_log.mean()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    strat_bins = pd.qcut(target_log, q=10, labels=False, duplicates="drop")

    for col in cat_cols:
        col_name = col if isinstance(col, str) else "_".join(col)
        if isinstance(col, list):
            df_tr[col_name] = df_tr[col].astype(str).agg("_".join, axis=1)
            df_te[col_name] = df_te[col].astype(str).agg("_".join, axis=1)

        # Initialize OOF columns
        for stat in ["mean", "median", "q25", "q75", "p10", "p90", "std"]:
            df_tr[f"TE_{col_name}_{stat}"] = np.nan
            df_te[f"TE_{col_name}_{stat}"] = np.nan

        # OOF Loop
        for fold, (train_idx, val_idx) in enumerate(skf.split(df_tr, strat_bins)):
            tr_fold = df_tr.iloc[train_idx]
            y_fold = target_log.iloc[train_idx]

            fold_df = pd.DataFrame({col_name: tr_fold[col_name], "y": y_fold})
            grp = fold_df.groupby(col_name)["y"]

            counts = grp.count()
            means = grp.mean()
            medians = grp.median()
            q25s = grp.quantile(0.25)
            q75s = grp.quantile(0.75)
            p10s = grp.quantile(0.10)
            p90s = grp.quantile(0.90)
            stds = grp.std().fillna(0.0)

            # Smooth mean encoding
            smooth_mean = (counts * means + smoothing * global_mean) / (counts + smoothing)

            val_keys = df_tr.iloc[val_idx][col_name]
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_mean"] = val_keys.map(smooth_mean).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_median"] = val_keys.map(medians).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_q25"] = val_keys.map(q25s).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_q75"] = val_keys.map(q75s).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_p10"] = val_keys.map(p10s).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_p90"] = val_keys.map(p90s).fillna(global_mean)
            df_tr.loc[df_tr.index[val_idx], f"TE_{col_name}_std"] = val_keys.map(stds).fillna(0.0)

        # Full train mapping for test set
        full_df = pd.DataFrame({col_name: df_tr[col_name], "y": target_log})
        grp_full = full_df.groupby(col_name)["y"]
        counts_full = grp_full.count()
        means_full = grp_full.mean()
        medians_full = grp_full.median()
        q25s_full = grp_full.quantile(0.25)
        q75s_full = grp_full.quantile(0.75)
        p10s_full = grp_full.quantile(0.10)
        p90s_full = grp_full.quantile(0.90)
        stds_full = grp_full.std().fillna(0.0)

        smooth_mean_full = (counts_full * means_full + smoothing * global_mean) / (counts_full + smoothing)

        test_keys = df_te[col_name]
        df_te[f"TE_{col_name}_mean"] = test_keys.map(smooth_mean_full).fillna(global_mean)
        df_te[f"TE_{col_name}_median"] = test_keys.map(medians_full).fillna(global_mean)
        df_te[f"TE_{col_name}_q25"] = test_keys.map(q25s_full).fillna(global_mean)
        df_te[f"TE_{col_name}_q75"] = test_keys.map(q75s_full).fillna(global_mean)
        df_te[f"TE_{col_name}_p10"] = test_keys.map(p10s_full).fillna(global_mean)
        df_te[f"TE_{col_name}_p90"] = test_keys.map(p90s_full).fillna(global_mean)
        df_te[f"TE_{col_name}_std"] = test_keys.map(stds_full).fillna(0.0)

    return df_tr, df_te

te_keys = [
    "Spec_BaseClass",
    "ProductConfigID",
    "VendorPartnerID",
    "ModelToken",
    "RegionCode",
    "Cat_Base_Year",
    "Cat_Vendor_Base",
    "Cat_Prod_Region"
]

df_tr_final, df_te_final = apply_oof_target_encoding(df_tr_stat, df_te_stat, cat_cols=te_keys)
print("✅ Out-of-fold target encoding complete!")


<a id="10"></a>
# 10. Level-1 Base Models Training (10-Fold CV across 7 Model Families)


In [ ]:
# Define feature subsets & categorical columns
drop_cols = ["TargetValue", "TransactionDate"] + [c for c in df_tr_final.columns if c.startswith("TE_") and c.endswith("_raw")]
features = [c for c in df_tr_final.columns if c not in drop_cols]

cat_cols = df_tr_final[features].select_dtypes(include=["object", "category"]).columns.tolist()

# Ordinal Encoding for Tree Models
for col in cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    df_tr_final[col] = oe.fit_transform(df_tr_final[[col]].astype(str))
    df_te_final[col] = oe.transform(df_te_final[[col]].astype(str))

# Impute NaN values for Linear Models & Tree compatibility
X_tr = df_tr_final[features].copy().fillna(-999)
X_te = df_te_final[features].copy().fillna(-999)
y_tr = np.log1p(df_tr_final["TargetValue"].clip(lower=0))

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

print(f"📊 Final Model Features: {len(features)}")
print(f"📊 Matrix Dimensions  : Train {X_tr.shape}, Test {X_te.shape}")


In [ ]:
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
strat_bins = pd.qcut(y_tr, q=10, labels=False, duplicates="drop")

models = {
    "lgb": lgb.LGBMRegressor(n_estimators=1800, learning_rate=0.03, num_leaves=63, subsample=0.8, colsample_bytree=0.7, random_state=42, verbose=-1, n_jobs=-1),
    "cat": CatBoostRegressor(iterations=1800, learning_rate=0.03, depth=7, random_seed=42, verbose=0, thread_count=-1),
    "xgb": xgb.XGBRegressor(n_estimators=1600, learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.7, random_state=42, n_jobs=-1),
    "et": ExtraTreesRegressor(n_estimators=300, max_depth=18, min_samples_split=5, random_state=42, n_jobs=-1),
    "ridge": Ridge(alpha=25.0),
    "lasso": Lasso(alpha=0.001, random_state=42),
    "elastic": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42)
}

oof_preds = {name: np.zeros(len(df_tr_final)) for name in models}
test_preds = {name: np.zeros(len(df_te_final)) for name in models}

print("🚀 Starting 10-Fold Level-1 Training across 7 Base Model Families...")
start_time = time.time()

for model_name, model in models.items():
    print(f"\n--- Training Family: {model_name.upper()} ---")
    fold_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr, strat_bins)):
        if model_name in ["ridge", "lasso", "elastic"]:
            X_fold_tr, y_fold_tr = X_tr_scaled[tr_idx], y_tr.iloc[tr_idx]
            X_fold_val, y_fold_val = X_tr_scaled[val_idx], y_tr.iloc[val_idx]
            X_test_input = X_te_scaled
        else:
            X_fold_tr, y_fold_tr = X_tr.iloc[tr_idx], y_tr.iloc[tr_idx]
            X_fold_val, y_fold_val = X_tr.iloc[val_idx], y_tr.iloc[val_idx]
            X_test_input = X_te

        model.fit(X_fold_tr, y_fold_tr)
        val_pred = model.predict(X_fold_val)
        oof_preds[model_name][val_idx] = val_pred
        test_preds[model_name] += model.predict(X_test_input) / n_splits
        
        rmse = np.sqrt(mean_squared_error(y_fold_val, val_pred))
        fold_scores.append(rmse)

    cv_rmse = np.mean(fold_scores)
    print(f"✅ {model_name.upper()} 10-Fold OOF RMSLE: {cv_rmse:.5f}")

print(f"\n🎉 All 7 Base Models Trained in {(time.time() - start_time)/60:.2f} minutes!")


<a id="11"></a>
# 11. Level-2 Meta-Learner Stacking & Nelder-Mead Optimization


In [ ]:
# Stack OOF and Test predictions
oof_matrix = pd.DataFrame(oof_preds)
test_matrix = pd.DataFrame(test_preds)

# Level-2 Meta Learner: RidgeCV
meta_ridge = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=10)
meta_ridge.fit(oof_matrix, y_tr)

meta_oof_ridge = meta_ridge.predict(oof_matrix)
meta_test_ridge = meta_ridge.predict(test_matrix)
score_ridge = np.sqrt(mean_squared_error(y_tr, meta_oof_ridge))

print(f"🎯 Level-2 RidgeCV Meta-Learner OOF RMSLE: {score_ridge:.5f}")
print("Ridge Meta Weights:", dict(zip(models.keys(), meta_ridge.coef_)))

# Non-Negative Nelder-Mead Weight Optimization
def loss_func(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    blend = np.dot(oof_matrix.values, weights)
    return np.sqrt(mean_squared_error(y_tr, blend))

init_weights = [1.0 / len(models)] * len(models)
bounds = [(0, 1)] * len(models)

res = minimize(loss_func, init_weights, method="Nelder-Mead", bounds=bounds)
opt_weights = res.x / np.sum(res.x)

meta_oof_opt = np.dot(oof_matrix.values, opt_weights)
meta_test_opt = np.dot(test_matrix.values, opt_weights)
score_opt = np.sqrt(mean_squared_error(y_tr, meta_oof_opt))

print(f"🎯 Nelder-Mead Optimal Ensemble OOF RMSLE : {score_opt:.5f}")
print("Optimal Model Weights:", dict(zip(models.keys(), np.round(opt_weights, 4))))

# Select best meta-predictions
if score_opt < score_ridge:
    final_oof = meta_oof_opt
    final_test = meta_test_opt
    best_stack_score = score_opt
    print("🏆 Selected Nelder-Mead Optimal Stacking Blend!")
else:
    final_oof = meta_oof_ridge
    final_test = meta_test_ridge
    best_stack_score = score_ridge
    print("🏆 Selected RidgeCV Meta-Learner Stacking Blend!")


<a id="12"></a>
# 12. Model Performance Comparison & Feature Importance Analysis


In [ ]:
perf_metrics = []
for name in models:
    score = np.sqrt(mean_squared_error(y_tr, oof_preds[name]))
    mae = mean_absolute_error(y_tr, oof_preds[name])
    r2 = r2_score(y_tr, oof_preds[name])
    perf_metrics.append({"Model": name.upper(), "OOF RMSLE": score, "MAE": mae, "R2 Score": r2})

perf_metrics.append({
    "Model": "PINNACLE ENSEMBLE (v20)",
    "OOF RMSLE": best_stack_score,
    "MAE": mean_absolute_error(y_tr, final_oof),
    "R2 Score": r2_score(y_tr, final_oof)
})

df_perf = pd.DataFrame(perf_metrics).sort_values(by="OOF RMSLE")
print(df_perf.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=df_perf, x="OOF RMSLE", y="Model", palette="viridis")
plt.title("Model Family Performance Comparison (10-Fold OOF RMSLE)")
plt.xlim(df_perf["OOF RMSLE"].min() - 0.005, df_perf["OOF RMSLE"].max() + 0.005)
plt.show()


<a id="13"></a>
# 13. Final Submission Verification & Distribution Inspection


In [ ]:
final_pred_values = np.expm1(final_test)
final_pred_values = np.clip(final_pred_values, a_min=0, a_max=None)

sub = pd.DataFrame({
    "TransactionID": df_test_raw["TransactionID"],
    "TargetValue": final_pred_values
})

sub.to_csv("submission.csv", index=False)
print("✅ submission.csv successfully saved!")
print(f"📊 Submission Shape: {sub.shape}")
print(f"📊 Target Summary Statistics:\n{sub['TargetValue'].describe()}")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(np.log1p(sub["TargetValue"]), kde=True, color="purple", bins=50)
ax.set_title("Distribution of Final Submission Log Predictions")
plt.show()


<a id="14"></a>
# 14. Conclusion & Key Insights

1. **Feature Mastery Impact**: Incorporating depreciation velocity, group baseline distance metrics, complexity flag counts, and 50-dim text embeddings provided key variance reduction across all 7 base model families.
2. **Compound Target Encodings**: High-order target encodings with multi-percentile statistics captured complex non-linear price variations across specific make/year/region clusters.
3. **Pinnacle Ensemble Synergy**: Combining 7 distinct model families with Nelder-Mead optimization provided strong cross-model generalization, lowering validation RMSLE.

